In [ ]:
import os, pickle

from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from datasets import balanced
from models import carl

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import hist

import torch
from torch.utils.data import Dataset
import lightning

In [ ]:
torch.set_float32_matmul_precision('high')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
})


In [3]:
OUTPUT_DIR = 'run/h4l/carl/'
SCALER_FILE_X = 'scaler.pkl'
SAMPLE_DIR = 'data/'

TRAIN_EPOCH = '69'
VAL_LOSS = '0.68'
VERSION = '0'

LOGS_DIR = f'{OUTPUT_DIR}/lightning_logs/version_{VERSION}'
CHECKPOINT_DIR = f'{LOGS_DIR}/checkpoints/'
CHECKPOINT = f'epoch={TRAIN_EPOCH}-val_loss={VAL_LOSS}.ckpt'

FEATURES=['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
          'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
          'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
          'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

BATCH_SIZE = 1024
SEED = 42

In [4]:
with open(f'{OUTPUT_DIR}/scaler.pkl', 'rb') as f:
    scaler_X = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_train.pkl', 'rb') as f:
    events_numerator_train = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_train.pkl', 'rb') as f:
    events_denominator_train = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_val.pkl', 'rb') as f:
    events_numerator_val = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_val.pkl', 'rb') as f:
    events_denominator_val = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_test.pkl', 'rb') as f:
    events_numerator_test = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_test.pkl', 'rb') as f:
    events_denominator_test = pickle.load(f)

training_data = balanced.BalancedDataset(events_numerator_train, events_denominator_train, FEATURES)
validation_data = balanced.BalancedDataset(events_numerator_val, events_denominator_val, FEATURES)
testing_data = balanced.BalancedDataset(events_numerator_test, events_denominator_test, FEATURES)

training_data.X = scaler_X.transform(training_data.X)
validation_data.X = scaler_X.transform(validation_data.X)
testing_data.X = scaler_X.transform(testing_data.X)

In [11]:
loaded_model = carl.CARL.load_from_checkpoint(os.path.join(CHECKPOINT_DIR, CHECKPOINT))

dl_train = torch.utils.data.DataLoader(training_data, batch_size=BATCH_SIZE)
dl_val = torch.utils.data.DataLoader(validation_data, batch_size=BATCH_SIZE)
dl_test = torch.utils.data.DataLoader(testing_data, batch_size=BATCH_SIZE)

trainer = lightning.Trainer(accelerator='gpu', devices=1)

# predictions_train = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_train]), axis=0).detach().numpy()
predictions_val = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_val]), axis=0).detach().numpy()
predictions_test = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_test]), axis=0).detach().numpy()

# targets_train = training_data.s
targets_val = validation_data.s
targets_test = testing_data.s

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [12]:
s_nbins = 50
s_min, s_max = 0.0, 1.0
s_step = (s_max - s_min)/s_nbins

In [13]:
p = predictions_val
t = targets_val

pred_binned = [p[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]
targets_binned = [t[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]

sig_per_bin = np.array([np.sum(targets_binned[i]==1.0) for i in range(s_nbins)])
bkg_per_bin = np.array([np.sum(targets_binned[i]==0.0) for i in range(s_nbins)])

s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)

sig_err = np.array([np.std(targets_binned[i]==1.0) for i in range(s_nbins)])
bkg_err = np.array([np.std(targets_binned[i]==0.0) for i in range(s_nbins)])
s_err = np.sqrt((sig_err * bkg_per_bin/(sig_per_bin + bkg_per_bin)**2)**2 + (-bkg_err*sig_per_bin/(sig_per_bin + bkg_per_bin)**2)**2)

s_centers = [s_min+(i+1/2)*s_step for i in range(s_nbins)]


/tmp/ipykernel_2434460/274831730.py:10: RuntimeWarning: invalid value encountered in divide
  s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [14]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(5,5), sharex=True)
plt.subplots_adjust(wspace=0, hspace=0)
ax1.set_aspect('equal', adjustable='box')

ax1.errorbar(s_centers, s_centers, color='black', linestyle='--', label='$\mathrm{MC}$ $\mathrm{estimate}$')
ax1.errorbar(s_centers, s_true, yerr=s_err, color='royalblue', marker='o', markersize=4, linestyle='none', label='$\mathrm{NSBI}$ $\mathrm{estimate}$')

ax1.set_ylim(s_min, s_max)
ax1.set_ylabel('$s(x) = \\frac{p_{q\\bar{q}}(x)}{ p_{q\\bar{q}}(x) + p_{gg}(x) }$')

ax1.legend(frameon=False)

ax2.errorbar(s_centers, np.array(s_centers)-np.array(s_centers), color='black', linestyle='--')
ax2.errorbar(s_centers, np.array(s_true)-np.array(s_centers), yerr=0., color='royalblue', marker='o', markersize=4, linestyle='none')

ax2.set_xlim(s_min, s_max)
ax2.set_xlabel('$\\hat{s}(x)$')
ax2.set_ylabel('$\\mathrm{Residual}$')
ax2.set_ylim(-0.1, 0.1)

plt.tight_layout()

fig.canvas.draw()  # update positions
ax1_pos, ax2_pos = ax1.get_position(), ax2.get_position()
ax2.set_position([ax1_pos.x0, ax2_pos.y0, ax1_pos.width, ax2_pos.height]) # align 2nd x-axis with 1st

plt.savefig('carl_calibration.pdf', bbox_inches='tight')
plt.close()